# 3.4 T.Pipelined 软件流水

## 概述

上一节我们对 ReLU 算子的实现进行了 tiling 分块与分核，在每个核上有多轮 tile，会循环多次进行拷入-计算-拷出，这个过程都是串行进行的。

因为拷入-计算-拷出处于不同的流水，本章我们介绍一个算子开发中常见的优化手段：多 buffer 流水排布，通俗地讲就是如果一块正在计算时能搬入后续数据，就可能减少等待。

## 学习目标

- 认识本例的 MTE2、Vector、MTE3 流水，区分同一 tile 的数据依赖与不同 tile 的交叠。
- 在 3.3 的 ReLU 实现上增加 `T.Pipelined` 和 buffer 版本配置，说明双 buffer 如何交替使用。
- 验证流水实现的正确性，并计算多 buffer 占用的 UB 空间。
- 使用 msprof 采集 kernel 耗时，对比单、双、三 buffer 的性能。

[本章导航](README.md) · [上一节](03.03_tiling.ipynb) · [下一节](03.05_execution_modes.ipynb)

<p><small>运行前完成第 1.4 节的环境准备，并使用本章 README 指定的已验证运行入口启动新内核。每页独立执行，工作目录为本章根目录；内核本身不初始化 NPU。</small></p>

## 1. 双buffer流水排布

昇腾 AI Core 内部有承担不同工作的指令队列，通常称为“流水”。本节 ReLU 主要涉及以下三条；它们与上一节的三段式代码一一对应。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">流水</th>
<th style="text-align: left; vertical-align: top;">本例承担的工作</th>
<th style="text-align: left; vertical-align: top;">对应代码</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">MTE2</td>
<td style="text-align: left; vertical-align: top;">将输入从 GM 拷入 UB</td>
<td style="text-align: left; vertical-align: top;"><code>T.copy(A[begin:begin+tile], a)</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Vector（V）</td>
<td style="text-align: left; vertical-align: top;">读取 UB 中的输入，执行 ReLU，结果存入 UB</td>
<td style="text-align: left; vertical-align: top;"><code>vld → vmax → vsts</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">MTE3</td>
<td style="text-align: left; vertical-align: top;">将结果从 UB 拷出到 GM</td>
<td style="text-align: left; vertical-align: top;"><code>T.copy(c, C[begin:begin+tile])</code></td>
</tr>
</tbody>
</table>

对同一个 tile，必须先完成拷入才能计算，完成计算才能拷出。不同流水可以独立推进，因此可以让**不同 tile** 的这三个阶段交叠。它与上一节的分核配合使用：分核让多个核同时处理不同任务；流水优化则让一个核内部的搬运与计算更充分地并行。

上一节按拷入、计算、拷出的顺序编写循环，`a` 和 `c` 各只有一个 UB 内存。后续轮次复用同一内存时，需要等待前面的使用者结束。双 buffer 为 `a` 和 `c` **各准备两个 UB 内存**，让相邻轮次使用不同位置，为提前搬入后续输入创造条件。两份 buffer 不是两个核，也不是把整个输入 Tensor 复制两遍。

![ReLU 的流水交叠与双 buffer](images/03.04_pipeline.png)

上半部分用三行表示拷入、计算、拷出可能交叠的关系，灰带对应下半部分的 UB 状态：a0 拷入 tile 2，Vector 从 a1 读取 tile 1 并将结果写入 c1，c0 正在拷出 tile 0。输入和输出各有两个 UB 内存空间。

图中的 tile 0、1、2 是**同一核依次负责的三个局部 tile**。沿用 3.3 的连续区间分配时，令 `rounds = N // (blocks * tile)`，其全局 tile 编号分别是 `bx * rounds`、`bx * rounds + 1`、`bx * rounds + 2`。

第 `it` 轮使用 `it % 2` 号 UB 内存；a0 中 tile 0 的输入被 Vector 读完后，便可接收 tile 2，不必等 c0 的 tile 0 结果写回结束。

UB 内存的复用仍需同步：拷入完成前不能读取 a，计算完成前不能拷出 c；旧输入读完前不能覆盖 a，旧结果拷出前不能覆盖 c。本例由编译器生成相应的依赖同步。循环开始时要先准备数据，结束时还要完成最后的写回，所以图中的稳定交叠不会覆盖整个执行过程。

三条流水的职责和并行关系可参阅CANN 社区文档[开启 DoubleBuffer](https://gitcode.com/cann/asc-devkit/blob/master/docs/zh/guide/operator_practice/simd_operator_optimization/pipeline_scheduling/enable_double_buffer.md)。

## 2. 为 ReLU 增加多 buffer 流水

保留 3.3 的 `begin = bx * (N // blocks) + it * tile`、ReLU 计算和输入输出接口，只在分配 UB 后配置 buffer 数量，并把外层 tile 循环换成 `T.Pipelined`。下面先配置双 buffer：

```python
import tilelang
import tilelang.language as T

N, tile, blocks, stages = 1179648, 1024, 8, 2


@T.prim_func
def main(A: T.Tensor((N,), "float32"), C: T.Tensor((N,), "float32")):
    with T.Kernel(blocks) as bx:
        a = T.alloc_shared((tile,), "float32")
        c = T.alloc_shared((tile,), "float32")
        T.annotate_buffer_versions({a: stages, c: stages})

        for it in T.Pipelined(N // (blocks * tile), num_stages=stages):
            begin = bx * (N // blocks) + it * tile
            T.copy(A[begin : begin + tile], a)

            with T.SimdVF():
                mask = T.simd.pset(32)
                zero = T.simd.vdup(0.0, "float32", mask)
                for group in T.serial(tile // 64):
                    ra = T.simd.vld(a[group * 64])
                    rc = T.simd.vmax(ra, zero, mask)
                    T.simd.vsts(c[group * 64], rc, mask)

            T.copy(c, C[begin : begin + tile])


kernel = tilelang.compile(main, out_idx=-1)
```

本节将循环的流水配置和各 buffer 的内存份数都设为 `stages`，按完整配置对比 1、2、3。

`N=1179648` 时，每个 block 处理 `1179648 / (8 × 1024) = 144` 轮 tile。固定 N、tile、blocks 后，改变 buffer 配置不会改变每个核负责的元素，也不会改变 ReLU 公式。完整实现和边界检查见 [vector_kernels.py](src/vector_kernels.py) 中的 `unary_pipeline`。

### T.Pipelined：安排不同轮次的搬运与计算

`T.Pipelined` 用于描述可以进行流水排布的循环。与 `T.serial` 相比，循环仍然要处理相同的 tile，区别在于编译器可以根据数据依赖，安排不同轮次的操作在不同硬件流水上并行执行。

在本例中，每轮包含三个操作：MTE2 将输入从 GM 搬入 a，Vector 读取 a 并将 ReLU 结果写入 c，MTE3 将 c 搬回 GM。同一轮必须满足“拷入完成后才能计算，计算完成后才能拷出”的依赖；不同轮次处理不同数据，只要使用的 UB 内存没有冲突，就有机会并行。例如，Vector 计算第 0 轮时，MTE2 可以向另一份输入 UB 内存搬入第 1 轮的数据。

`T.Pipelined(N // (blocks * tile), num_stages=stages)` 中，第一个参数是每个核处理的 tile 轮数；`num_stages` 给出编译器安排该循环时使用的流水深度配置。编译器结合操作间的数据依赖、buffer 配置和存储容量安排执行顺序，并处理必要的同步。源代码仍按“拷入—计算—拷出”书写，不需要手工把第 1 轮的拷入插到第 0 轮的计算前后。

### T.annotate_buffer_versions：为流水提供可轮换的 UB 内存

`T.annotate_buffer_versions({a: stages, c: stages})` 是编译期的 buffer 配置：字典的键是要配置的 Tensor，值是为它指定的 UB 内存份数。本例 `stages=2`，表示输入 a 和输出 c **各使用两份独立的 UB 内存**。它不会把输入数据自动复制两遍；每轮仍由对应的 `T.copy` 搬入新的输入，由 Vector 计算生成新的输出。

代码中仍然使用 a、c 这两个名字，编译器负责扩展存储并将不同轮次的访问映射到对应的 UB 内存。对于本例从 0 开始的循环，可以按 `it % 2` 理解双 buffer 的轮换：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">tile 轮次 it</th>
<th style="text-align: left; vertical-align: top;">输入 a 使用的 UB 内存</th>
<th style="text-align: left; vertical-align: top;">输出 c 使用的 UB 内存</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">a 的内存 0</td>
<td style="text-align: left; vertical-align: top;">c 的内存 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">a 的内存 1</td>
<td style="text-align: left; vertical-align: top;">c 的内存 1</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">复用 a 的内存 0</td>
<td style="text-align: left; vertical-align: top;">复用 c 的内存 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">3</td>
<td style="text-align: left; vertical-align: top;">复用 a 的内存 1</td>
<td style="text-align: left; vertical-align: top;">复用 c 的内存 1</td>
</tr>
</tbody>
</table>

这里的编号分别属于 a 和 c；a 的内存 0 与 c 的内存 0 不是同一块内存。表格描述每轮使用哪份内存，不表示同一行的搬运和计算会同时发生。

### 两者如何配合

`T.Pipelined` 描述循环的流水排布，`T.annotate_buffer_versions` 显式指定各 buffer 可使用的内存份数：前者安排不同轮次的操作，后者为这些操作并行提供存储条件。本节将两处配置都设为 `stages`，便于对比单、双、三 buffer，但两项配置控制的对象不同。

以输入 a 为例，如果只有一份内存，下一轮拷入必须等待上一轮 Vector 读取结束，否则会覆盖尚未使用完的数据。配置双 buffer 后，第 1 轮可以写入 a 的内存 1，同时第 0 轮读取 a 的内存 0。

到第 2 轮重新使用 a 的内存 0 时，仍要等第 0 轮读取结束。输出 c 也一样：Vector 再次写入某份内存前，要等此前的 MTE3 拷出完成。这些依赖由后端分析并通过同步指令保证；增加 buffer 可以减少因共用内存产生的等待，但不能删除必要的等待。

整个循环开始时，要先搬入第一块数据，计算流水才能开始工作；中间轮次有机会形成搬入、计算、搬出的并行；最后还要等剩余结果写回。这就是流水的填充、稳定运行和排空过程。更多 buffer 会增加 UB 占用，是否能进一步减少等待，还取决于各流水耗时和可并行的工作量。下一节先检查正确性与 UB 用量，再通过 kernel 耗时比较优化效果。

## 3. 验证正确性与 UB 用量

先沿用前面的四类输入：随机值、全零、常数和包含正负数的 ramp。对串行基线以及流水配置 1、2、3 分别验证完整输出，共 16 组检查；ReLU 使用 `rtol=0, atol=0`，要求逐元素一致。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">实现</th>
<th style="text-align: left; vertical-align: top;">N / tile / blocks</th>
<th style="text-align: left; vertical-align: top;">输入类型</th>
<th style="text-align: left; vertical-align: top;">验证要求</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.serial</code>，单 buffer</td>
<td style="text-align: left; vertical-align: top;">1179648 / 1024 / 8</td>
<td style="text-align: left; vertical-align: top;">四类输入</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过，误差为 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.Pipelined</code>，配置 1</td>
<td style="text-align: left; vertical-align: top;">同上</td>
<td style="text-align: left; vertical-align: top;">四类输入</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过，误差为 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.Pipelined</code>，配置 2</td>
<td style="text-align: left; vertical-align: top;">同上</td>
<td style="text-align: left; vertical-align: top;">四类输入</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过，误差为 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.Pipelined</code>，配置 3</td>
<td style="text-align: left; vertical-align: top;">同上</td>
<td style="text-align: left; vertical-align: top;">四类输入</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过，误差为 0</td>
</tr>
</tbody>
</table>

**正确性验证结果**：串行和三个流水配置共 16 组完整输出校验全部通过，ReLU 最大绝对误差均为 0。

多 buffer 需要额外的 UB。延续 3.2 的容量计算，本例只有输入 a 和输出 c，两者都是 fp32，所以每个核的这两份 buffer 合计占用：

`UB 字节数 = 2 × tile × 4 × versions`

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">每个 tile 的元素数</th>
<th style="text-align: left; vertical-align: top;">单 buffer</th>
<th style="text-align: left; vertical-align: top;">双 buffer</th>
<th style="text-align: left; vertical-align: top;">三 buffer</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">1024（本节实验）</td>
<td style="text-align: left; vertical-align: top;">8 KiB</td>
<td style="text-align: left; vertical-align: top;">16 KiB</td>
<td style="text-align: left; vertical-align: top;">24 KiB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">64 KiB</td>
<td style="text-align: left; vertical-align: top;">128 KiB</td>
<td style="text-align: left; vertical-align: top;">192 KiB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">16384</td>
<td style="text-align: left; vertical-align: top;">128 KiB</td>
<td style="text-align: left; vertical-align: top;">256 KiB，超出</td>
<td style="text-align: left; vertical-align: top;">384 KiB，超出</td>
</tr>
</tbody>
</table>

当前环境可分配的UB容量为 248 KiB。这里按本例的 a、c 计算；扩展算子若增加中间 buffer，也要计入容量。`tile=16384, versions=2` 会超出限制；这说明 tile 大小，在增加 buffer 后需要重新计算检查。

In [1]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)

def run_checked(script):
    try:
        result = subprocess.run([sys.executable, script], check=True,
                                capture_output=True, text=True)
    except subprocess.CalledProcessError as error:
        print(error.stdout)
        print(error.stderr)
        raise
    log_name = Path(script).stem + "_notebook.log"
    Path(OUTPUT_DIR / "validation", log_name).write_text(result.stdout + result.stderr)
    print("\n".join(line for line in result.stdout.splitlines()
                    if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

run_checked("src/run_pipeline.py")

Checking ReLU mode='serial' random
ReLU mode='serial' random: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode='serial' zero
ReLU mode='serial' zero: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode='serial' constant
ReLU mode='serial' constant: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode='serial' ramp
ReLU mode='serial' ramp: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode=1 random
ReLU mode=1 random: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode=1 zero
ReLU mode=1 zero: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode=1 constant
ReLU mode=1 constant: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode=1 ramp
ReLU mode=1 ramp: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode=2 random
ReLU mode=2 random: full-output PASS; shape=(1179648,); max_abs_error=0
Checking ReLU mode=2 zero
ReLU mode=2 zero: full-out

## 4. 用 msprof 比较 kernel 性能

直接沿用 3.3 的同一个性能用例及其 8 核串行基线，性能以 msprof 导出的 **`Task Duration(us)`** 为准。保持 ReLU、fp32、`N=1179648`、`tile=1024`、`blocks=8` 不变，比较上一节的 `T.serial` 与本节的流水配置 1、2、3。

每个配置先运行一次并校验结果，再运行三批，每批预热 10 次、采样 30 次，共保留 90 个目标 kernel 样本。每次调用后同步，每批结束后校验完整输出；输入复用同一份 ramp，不主动清缓存。只统计 `Op Name=main_kernel`、`Task Type=AI_VECTOR_CORE`、`Block Num=8` 的记录，排除预热和精度检查相关的其他算子。

这里的耗时是设备上的 kernel 执行时间，不计 Python 调用、输出分配、Host 同步耗时，也不把 `Task Wait Time` 加进来。此前按同一协议采集了以下设备耗时，并计算加速比。

统一性能用例与 3.3 相同：`shape=(1179648,)`、`tile=1024`、`blocks=8`，每核 144 轮。以下数据为此前寄存器微指令实现的测量记录。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">实现</th>
<th style="text-align: left; vertical-align: top;">UB 用量</th>
<th style="text-align: left; vertical-align: top;">三批中位数/μs</th>
<th style="text-align: left; vertical-align: top;">kernel 中位数/μs</th>
<th style="text-align: left; vertical-align: top;">相对8核串行加速比</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">T.serial，单 buffer</td>
<td style="text-align: left; vertical-align: top;">8 KiB</td>
<td style="text-align: left; vertical-align: top;">31.7430, 31.7160, 31.6875</td>
<td style="text-align: left; vertical-align: top;">31.7215</td>
<td style="text-align: left; vertical-align: top;">1.00×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">T.Pipelined，配置 1</td>
<td style="text-align: left; vertical-align: top;">8 KiB</td>
<td style="text-align: left; vertical-align: top;">31.7050, 31.6795, 31.6655</td>
<td style="text-align: left; vertical-align: top;">31.6895</td>
<td style="text-align: left; vertical-align: top;">1.00×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">T.Pipelined，配置 2</td>
<td style="text-align: left; vertical-align: top;">16 KiB</td>
<td style="text-align: left; vertical-align: top;">16.7930, 16.8300, 16.8105</td>
<td style="text-align: left; vertical-align: top;">16.8070</td>
<td style="text-align: left; vertical-align: top;">1.89×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">T.Pipelined，配置 3</td>
<td style="text-align: left; vertical-align: top;">24 KiB</td>
<td style="text-align: left; vertical-align: top;">11.3955, 11.3375, 11.3575</td>
<td style="text-align: left; vertical-align: top;">11.3635</td>
<td style="text-align: left; vertical-align: top;">2.79×</td>
</tr>
</tbody>
</table>

本例固定 8 核后，双 buffer 相对串行加速 **1.89 倍**，三 buffer 加速 **2.79 倍**。这与 3.3 的分核对比形成递进：先在同一输入上由 1 核变为 8 核，再保持 8 核增加 buffer。UB 用量由 8 KiB 增至 16/24 KiB，收益以当前用例实测为准。

> 表中性能为课程参考测量记录，原始采集附件不在课程目录中。可按本节给出的命令重新采集当前环境的结果。

**复现采集**

在本章目录按以下命令依次运行，避免几组性能采集相互干扰：

```bash
source src/env.sh
for mode in serial 1 2 3; do
  msprof --output="${COURSE_OUTPUT_DIR}/profiles/msprof_register_revision/replay_${mode}" \
    --task-time=on --ai-core=off --runtime-api=on \
    python src/profile_pipeline.py "$mode" 1024
done
```

每组 `op_summary` 应含 121 行目标 kernel：1 行初始校验，随后 3 × 40 行。按启动时间排序，去掉第 1 行，再从每组 40 行中去掉前 10 行预热，汇总剩余的 90 个 `Task Duration(us)`。不同采集轮次存在正常波动，应比较自己的同批数据。



采集入口见 [profile_pipeline.py](src/profile_pipeline.py)。正文保留课程参考测量数据；执行上述命令后，可在指定的采集输出位置查看本次原始 CSV，并按前述规则提取 90 个有效样本。

## 小结

3.2 完成了 ReLU 功能，3.3 用 tiling 控制单次 UB 用量并将工作分配到多个核；本节继续在每个核内部增加 buffer，使不同 tile 的拷入、计算、拷出有机会交叠。完成实现后，先验证完整输出、检查 UB，再用 msprof 比较 kernel 耗时。下一节将用相同计算任务认识 SimdVF 与 SimtVF 的表达差异。

## 课后练习

1. **读图题**：图示时刻为什么 a0 可以接收 tile 2，而 c0 仍在拷出 tile 0？它们各自需要等待哪个使用者完成？
2. **容量题**：ReLU 的 `tile=8192`，输入 a 与输出 c 都采用三 buffer，需要多少 UB？如果改成 `tile=16384`、双 buffer，是否仍能放入当前平台的 UB？
3. **编程题**：为上一节的 Exp 练习增加多 buffer 流水。使用 `N=1179648, tile=1024, blocks=8`，测试配置 1、2、3；输入限定为有限 fp32、绝对值不超过 10，按四类用例检查完整输出，容差 `rtol=1e-5, atol=1e-6`。完成后可仿照本节用 msprof 比较性能。



先独立完成，再阅读下面的解析和 [Exp 完整答案](answer/03.04_pipeline_exp.py)。

Exp 答案的配置 1、2、3 均按四类输入完成 NPU 校验；完整输出满足指定容差。

<details>
<summary>展开练习解析</summary>

1. a0 和 c0 是两块独立的 UB 存储。a0 中 tile 0 的输入读完后可复用；c0 中 tile 0 的结果必须等 MTE3 拷出完成后才能覆盖。同一编号不要求它们始终保存同一轮数据。

2. 2 × 8192 × 4 × 3 = 192 KiB，未超过当前环境可分配的UB容量（248 KiB）。2 × 16384 × 4 × 2 = 256 KiB，超过容量。

3. Exp 保持相同的分核索引与搬运结构，将计算改为 `T.simd.vexp(ra, mask)`；为 a、c 配置同样的 buffer 份数。完整答案验证三个配置和四类输入，共 12 组完整输出。kernel 性能应另外用 msprof 测量。

</details>

In [2]:
run_checked("answer/03.04_pipeline_exp.py")

Checking Exp stages=1 random
Exp stages=1 random: full-output PASS; shape=(1179648,); max_abs_error=7.6293945e-06
Checking Exp stages=1 zero
Exp stages=1 zero: full-output PASS; shape=(1179648,); max_abs_error=0
Checking Exp stages=1 constant
Exp stages=1 constant: full-output PASS; shape=(1179648,); max_abs_error=0
Checking Exp stages=1 ramp
Exp stages=1 ramp: full-output PASS; shape=(1179648,); max_abs_error=3.8146973e-06
Checking Exp stages=2 random
Exp stages=2 random: full-output PASS; shape=(1179648,); max_abs_error=7.6293945e-06
Checking Exp stages=2 zero
Exp stages=2 zero: full-output PASS; shape=(1179648,); max_abs_error=0
Checking Exp stages=2 constant
Exp stages=2 constant: full-output PASS; shape=(1179648,); max_abs_error=0
Checking Exp stages=2 ramp
Exp stages=2 ramp: full-output PASS; shape=(1179648,); max_abs_error=3.8146973e-06
Checking Exp stages=3 random
Exp stages=3 random: full-output PASS; shape=(1179648,); max_abs_error=7.6293945e-06
Checking Exp stages=3 zero
Exp

[本章导航](README.md) · [上一节](03.03_tiling.ipynb) · [下一节](03.05_execution_modes.ipynb)